In [1]:
import os
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
import dagshub

# Set up MLflow
mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
mlflow.set_experiment("KNN Optimization")
os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'fbaccfb8cf4e8d2d195cd05e9a53dbfe32323695'

# Fetch experiment data
experiment_id = "KNN Optimization"  # Replace with your experiment ID

# Retrieve all runs from the specified MLflow experiment
runs = mlflow.search_runs(experiment_ids=[mlflow.get_experiment_by_name("KNN Optimization").experiment_id])

if runs.empty:
    print("No runs found in the specified experiment")

# Define the columns you want to select
params = [
    "params.n_neighbors", "params.weights", "params.algorithm", "params.leaf_size", "params.p"
]
metrics = ["metrics.train_mse", "metrics.test_mse"]
tags = ["tags.target_feature", "tags.feature_addition_rounds", "tags.feature_dropping_threshold"]

# Combine the metrics and parameter lists
columns_to_select = metrics + params + tags

df = runs[columns_to_select]

# Preprocess the data
df.sort_values(by='metrics.test_mse', ascending=False)
df.dropna()

df = df.dropna()

df[[
    "params.n_neighbors", "params.weights", "params.algorithm", "params.leaf_size", "params.p"
]].drop_duplicates()

no_duos_index = df[[
    "params.n_neighbors", "params.weights", "params.algorithm", "params.leaf_size", "params.p"
]].drop_duplicates(keep='last').index

# Replace "None" with np.nan for specified columns
df['params.n_neighbors'] = df['params.n_neighbors'].replace('None', np.nan)
df['params.weights'] = df['params.weights'].replace('None', np.nan)
df['params.algorithm'] = df['params.algorithm'].replace('None', np.nan)
df['params.leaf_size'] = df['params.leaf_size'].replace('None', np.nan)
df['params.p'] = df['params.p'].replace('None', np.nan)

# Convert the relevant columns to numeric
df['params.n_neighbors'] = pd.to_numeric(df['params.n_neighbors'], errors='coerce')
df['params.weights'] = df['params.weights'].astype('category')
df['params.algorithm'] = df['params.algorithm'].astype('category')
df['params.leaf_size'] = pd.to_numeric(df['params.leaf_size'], errors='coerce')
df['params.p'] = pd.to_numeric(df['params.p'], errors='coerce')

runs_df = df.copy()

import plotly.graph_objs as go
# Determine if a parameter should be plotted as a line or bar plot
def should_use_line_plot(param):
    continuous_params = [
        'params.n_neighbors', 'params.leaf_size', 'params.p'
    ]
    return param in continuous_params

# Create plots for each parameter - Train and Test MSE
for param in params:
    fig = go.Figure()
    
    grouped_test = df.groupby(param)['metrics.test_mse'].mean()
    grouped_train = df.groupby(param)['metrics.train_mse'].mean()
    
    if should_use_line_plot(param):
        fig.add_trace(go.Scatter(x=grouped_test.index, y=grouped_test, mode='lines+markers', name='Average Test MSE', line=dict(color='blue')))
        fig.add_trace(go.Scatter(x=grouped_train.index, y=grouped_train, mode='lines+markers', name='Average Train MSE', line=dict(color='red')))
    else:
        fig.add_trace(go.Bar(x=grouped_test.index, y=grouped_test, name='Average Test MSE', marker_color='blue'))
        fig.add_trace(go.Bar(x=grouped_train.index, y=grouped_train, name='Average Train MSE', marker_color='red'))

    # Update layout for better visualization
    fig.update_layout(
        title=f'Effect of {param} on Mean MSE',
        xaxis_title=param,
        yaxis_title='Mean Squared Error',
        yaxis=dict(range=[min(grouped_test.min(), grouped_train.min()) * 0.9, max(grouped_test.max(), grouped_train.max()) * 1.1]),
        barmode='group' if not should_use_line_plot(param) else None
    )

    fig.show()


Initialized MLflow to track repo "najibabounasr/MacroEconomicAPI"

Repository najibabounasr/MacroEconomicAPI initialized!